# CCE PoC v3: multi-model + semantic-entropy ablation

Runs the v3 PoC on a single LLM. Run this notebook **3 times**, once per model, by changing the `MODEL` cell. Each run takes ~2h on Colab Pro A100 and writes per-model artifacts to Drive. After all three runs complete, run the Aggregation cell at the bottom to produce the cross-model paper tables.

**What v3 adds over v2:**
- Multi-model support (CodeLlama, Qwen2.5-Coder, DeepSeek-Coder)
- Semantic entropy (Kuhn 2023 / Farquhar 2024) as a 7th feature group and 3 new ablation arms (`semantic_entropy_only`, `flare_plus_se`, `all_features`)
- Per-model output paths so runs don't clobber each other

**Wall-clock per model (A100):**
- Phase 1 (greedy + features):  ~25 min
- Phase 1.5 (sampled gens for SE): ~40 min
- Phase 3 (gated retrieval): ~25 min
- **Total ~90 min per model.**

## 1. Choose the model for this run

In [ ]:
# Change this and run all cells. After all three runs are complete,
# scroll to the bottom and run the aggregation cell.
MODEL = "codellama/CodeLlama-7b-Instruct-hf"
# MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
# MODEL = "deepseek-ai/deepseek-coder-7b-instruct-v1.5"
print(f'this run: {MODEL}')

## 2. Install + clone

In [ ]:
!pip install -q torch transformers accelerate bitsandbytes sentence-transformers scikit-learn

In [ ]:
import os
if not os.path.exists('/content/reposynth'):
    !git clone https://github.com/aniJani/reposynth.git /content/reposynth
%cd /content/reposynth
!git checkout Research
!git pull --rebase || true

## 3. HuggingFace login (CodeLlama and DeepSeek are gated; Qwen is not)

In [ ]:
from huggingface_hub import login
from google.colab import userdata
try:
    login(userdata.get('HF_TOKEN'))
except Exception as e:
    print('Set HF_TOKEN as a Colab secret first:', e)

## 4. Mount Drive (per-model caches survive disconnects)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT_DIR = '/content/drive/MyDrive/cce_poc_v3'
!mkdir -p {OUT_DIR}
print(f'will write to {OUT_DIR}')

## 5. Run the PoC for this model

Resumable: if Phase 1 features for this model are already cached on Drive, pass `--skip-generation`. Same for `--skip-se-collection`.

In [ ]:
%cd /content/reposynth
!python /content/reposynth/research/paper/cce_poc_v3.py \
    --model "{MODEL}" \
    --repos-dir /content \
    --out-dir   {OUT_DIR}

## 6. Inspect this model's results

In [ ]:
import json, re, glob
slug = re.sub(r'[^A-Za-z0-9]+', '_', MODEL).strip('_')
with open(f'{OUT_DIR}/results__{slug}.json') as f:
    res = json.load(f)

print(f"model = {res['model']}, n_tasks = {res['n_tasks']}\n")
print('PHASE 2 (LOO) F1:')
print(f"{'arm':<22} {'#feat':>5} {'acc':>5} {'prec':>5} {'rec':>5} {'f1':>5}")
for arm, r in res['phase2_loo_ablation'].items():
    print(f"{arm:<22} {r['n_features']:>5d} {r['accuracy']:>5.3f} "
          f"{r['precision']:>5.3f} {r['recall']:>5.3f} {r['f1']:>5.3f}")

if res.get('phase3_end_to_end'):
    print('\nPHASE 3 (end-to-end gating):')
    print(f"{'arm':<22} {'init':>5} {'final':>5} {'always':>5} {'used':>5} {'save%':>6}")
    for arm, r in res['phase3_end_to_end'].items():
        print(f"{arm:<22} {r['initial_accuracy']:>5.3f} {r['final_accuracy']:>5.3f} "
              f"{r['always_retrieve_accuracy']:>5.3f} {r['n_retrievals_used']:>5d} "
              f"{r['retrieval_save_rate']*100:>5.1f}%")

---
## 7. Aggregation across models *(run this cell ONCE after all three model runs complete)*

In [ ]:
%cd /content/reposynth
!python /content/reposynth/research/paper/cce_poc_v3_aggregate.py \
    --in-dir {OUT_DIR} \
    --out    {OUT_DIR}/v3_combined_results.json